SQLite 存储：SqliteSaver

In [3]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    count: int

def increment(state: SimpleState) -> dict:
    return {"count": state.get("count", 0) + 1}

# 创建图
graph = StateGraph(SimpleState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# 使用 with 语句创建 SQLite checkpointer（推荐方式）
with SqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
    app = graph.compile(checkpointer=checkpointer)
    
    # 使用方式与 MemorySaver 相同
    config = {"configurable": {"thread_id": "user-123"}}
    result = app.invoke({"count": 0}, config)
    print(result)

# 或者使用内存 SQLite（用于测试）
with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    app = graph.compile(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "user-456"}}
    result = app.invoke({"count": 0}, config)
    print(result)

{'count': 1}
{'count': 1}
